# Klassifikation mit allen Werten

`JobSat` als Zielwert


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [46]:

df = pd.read_csv("One-Hot-Encoded.csv")


bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

MaxAge                                                            float64
AgeNum                                                            float64
WorkExp                                                           float64
YearsCode                                                         float64
RemoteCategoryNum                                                 float64
                                                                   ...   
AIAgents_no, i use ai exclusively in copilot/autocomplete mode      int64
AIAgents_yes, i use ai agents at work daily                         int64
AIAgents_yes, i use ai agents at work monthly or infrequently       int64
AIAgents_yes, i use ai agents at work weekly                        int64
AIAgents_nan                                                        int64
Length: 484, dtype: object

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [47]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"

y = df["JobSat"].apply(map_jobsat)
y.value_counts()

JobSat
High      18623
Medium     5851
Low        1632
Name: count, dtype: int64

## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`



In [ ]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"] #?

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()


X.head()

Textspalten: ['LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIAgent_Uses']
Numerische Spalten: ['MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'CompTotal', 'ConvertedCompYearly', 'ConvertedCompTotal', 'MainBranch_i am a developer by profession', 'MainBranch_i am learning to code', 'MainBranch_i am not primarily a developer, but i write code sometimes as part of my work/studies', 'MainBranch_i code primarily as a hobby', 'MainBranch_i used to be a developer by profession, but no longer am', 'MainBranch_i work with developers or my work supports developers but am not a developer by profession', 'MainBranch_nan', 'Age_18-24 years old', 'Age_25-

,__text__,MaxAge,AgeNum,WorkExp,YearsCode,RemoteCategoryNum,CompTotal,ConvertedCompYearly,ConvertedCompTotal,MainBranch_i am a developer by profession,...,"AISelect_yes, i use ai tools monthly or infrequently","AISelect_yes, i use ai tools weekly",AISelect_nan,"AIAgents_no, and i don't plan to","AIAgents_no, but i plan to","AIAgents_no, i use ai exclusively in copilot/autocomplete mode","AIAgents_yes, i use ai agents at work daily","AIAgents_yes, i use ai agents at work monthly or infrequently","AIAgents_yes, i use ai agents at work weekly",AIAgents_nan
0,"['bash/shell (all shells)', 'dart', 'sql'] ['d...",34.0,29.0,8.0,14.0,0.00,52800.0,61256.0,61659.84,1,...,1,0,0,0,0,0,0,1,0,0
1,"['java'] ['java', 'python', 'swift'] ['dynamod...",34.0,29.0,2.0,10.0,0.25,90000.0,104413.0,105102.00,1,...,0,1,0,1,0,0,0,0,0,0
2,"['dart', 'html/css', 'javascript', 'typescript...",44.0,39.0,10.0,12.0,NaN,2214000.0,53061.0,NaN,1,...,0,0,0,0,0,0,0,0,1,0
3,"['java', 'kotlin', 'sql'] ['java', 'kotlin'] [...",44.0,39.0,4.0,5.0,0.00,31200.0,36197.0,36435.36,1,...,0,1,0,0,0,0,0,1,0,0
4,"['c', 'c#', 'c++', 'delphi', 'html/css', 'java...",44.0,39.0,21.0,22.0,NaN,60000.0,60000.0,60000.00,1,...,0,1,0,1,0,0,0,0,0,0


## Train/Test Split



In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 20884
Test size: 5222
Train class distribution:
 JobSat
High      0.713369
Medium    0.224143
Low       0.062488
Name: proportion, dtype: float64
Test class distribution:
 JobSat
High      0.713328
Medium    0.224052
Low       0.062620
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten




In [50]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

preprocessor

,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'


## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit L1-LinearSVC)
- Klassifikator (LinearSVC)


In [51]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(LinearSVC(penalty="l1", dual=False, C=0.5))),
    ("classifier", LinearSVC())
])

pipeline

,steps,"[('preprocessing', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## GridSearchCV



In [52]:

parameters = {
    'vectorizer__max_df': (0.5, 0.75, 1.0),
    'vectorizer__analyzer': ('word', 'char'),
    'feature_selection__threshold': (None, 'mean')
    #'classifier__kernel': ('linear', 'rbf')
}

grid_search = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)

grid

,estimator,Pipeline(step...LinearSVC())])
,param_grid,"{'classifier__C': [0.1, 1.0, ...], 'classifier__class_weight': [None, 'balanced'], 'feature_selection__estimator__C': [0.1, 0.5, ...], 'feature_selection__threshold': [None, 'mean'], ...}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('text', ...), ('num', ...)]"


## Grid Search + Beste Parameter


In [ ]:

grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits
[CV] END classifier__C=0.1, classifier__class_weight=None, feature_selection__estimator__C=0.1, feature_selection__threshold=None, preprocessing__text__max_df=0.75, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=   0.1s
[CV] END classifier__C=0.1, classifier__class_weight=None, feature_selection__estimator__C=0.1, feature_selection__threshold=None, preprocessing__text__max_df=0.75, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=   0.1s
[CV] END classifier__C=0.1, classifier__class_weight=None, feature_selection__estimator__C=0.1, feature_selection__threshold=None, preprocessing__text__max_df=0.75, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=   0.1s
[CV] END classifier__C=0.1, classifier__class_weight=None, feature_selection__estimator__C=0.1, feature_selection__threshold=None, preprocessing__text__max_df=0.75, pre

KeyboardInterrupt: 

## Evaluation auf Testdaten


In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

In [ ]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))